# U2T02: SimCSE — Train Your Own Sentence Embedding Model

Este notebook contiene la implementación completa para:
1. **Cargar y validar datos:** `snli_train_100k.jsonl` (unsupervised 165k oraciones, supervised 33k pares con hard negatives) y STS-B (dev/test).
2. **Verificar baselines de referencia:** Raw `bert-base-uncased` (59.31/47.29) y SBERT-2019 (80.77/76.98).
3. **Entrenar SimCSE Unsupervised & Supervised** con `bert-base-uncased`.
4. **Ejecutar ablaciones requeridas:** Mismo dropout mask (unsupervised) y Hard negatives OFF (supervised).
5. **Métricas de geometría:** Alignment y Uniformity (Wang & Isola, 2020).
6. **Exportar a Hugging Face Hub** en formato `sentence-transformers` con verificación de recarga.

In [ ]:
# Instalación de dependencias (para Google Colab o entorno nuevo)
!pip install -q torch transformers datasets sentence-transformers scipy scikit-learn matplotlib seaborn huggingface_hub accelerate

In [ ]:
import os
import sys
import json
import torch

# Añadir directorio raíz al path
if os.path.exists("src"):
    sys.path.append(".")
elif os.path.exists("../src"):
    sys.path.append("..")

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Dispositivo de cómputo activo: {device}")

## 1. Sanity Check de Baselines en STS-B
Validamos que el código de evaluación replique con exactitud los valores de referencia del PDF.

In [ ]:
from src.data_loader import load_stsb_data
from src.evaluate import evaluate_stsb, evaluate_sentence_transformer
from transformers import AutoModel, AutoTokenizer

dev_stsb, test_stsb = load_stsb_data()

# 1. Raw bert-base-uncased con mean pooling
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
raw_bert = AutoModel.from_pretrained("bert-base-uncased").to(device)

raw_dev = evaluate_stsb(raw_bert, tokenizer, dev_stsb, device=device, pooling="mean")
raw_test = evaluate_stsb(raw_bert, tokenizer, test_stsb, device=device, pooling="mean")
print(f"Raw BERT (mean): Dev={raw_dev['spearman']:.2f} (Ref: 59.31) | Test={raw_test['spearman']:.2f} (Ref: 47.29)")

# 2. SBERT-2019 (bert-base-nli-mean-tokens)
sbert_dev = evaluate_sentence_transformer("bert-base-nli-mean-tokens", dev_stsb)
sbert_test = evaluate_sentence_transformer("bert-base-nli-mean-tokens", test_stsb)
print(f"SBERT-2019:     Dev={sbert_dev['spearman']:.2f} (Ref: 80.77) | Test={sbert_test['spearman']:.2f} (Ref: 76.98)")

## 2. Validación de Datos: SNLI 100k
Verificamos que `snli_train_100k.jsonl` produzca exactamente 165,529 oraciones únicas y 33,351 pares supervisados.

In [ ]:
from src.data_loader import load_snli_100k

data_path = "snli_train_100k.jsonl"
if os.path.exists(data_path):
    unsupervised_sentences, supervised_items = load_snli_100k(data_path)
else:
    print(f"Por favor coloca el archivo '{data_path}' en la raíz del proyecto.")

## 3. Entrenamiento Unsupervised SimCSE (Eq. 1)
Entrenamos con aumento de datos basado en dropout independiente.

In [ ]:
!python -m src.train --mode unsupervised --data_path snli_train_100k.jsonl --batch_size 64 --lr 3e-5 --epochs 1 --seed 42

## 4. Ablación Unsupervised: Mismo Dropout Mask
Comprobamos el colapso de representación al usar la misma máscara de dropout ($z_1 = z_2$).

In [ ]:
!python -m src.train --mode unsupervised --data_path snli_train_100k.jsonl --same_dropout_ablation --batch_size 64 --lr 3e-5 --epochs 1 --seed 42

## 5. Entrenamiento Supervised SimCSE con Hard Negatives (Eq. 5)
Entrenamos con pares premisa-implicación y contradicciones como hard negatives.

In [ ]:
!python -m src.train --mode supervised --data_path snli_train_100k.jsonl --batch_size 64 --lr 5e-5 --epochs 3 --seed 42

## 6. Ablación Supervised: Hard Negatives OFF
Comprobamos el impacto de remover las contradicciones del denominador de la pérdida.

In [ ]:
!python -m src.train --mode supervised --data_path snli_train_100k.jsonl --no_hard_negatives --batch_size 64 --lr 5e-5 --epochs 3 --seed 42

## 7. Exportación y Publicación en Hugging Face Hub (Parte 7)
Exportamos el mejor modelo a formato `sentence-transformers`, lo subimos al Hub y verificamos la igualdad numérica recargándolo desde el Hub.

In [ ]:
# Ajusta tu usuario y token de Hugging Face
HF_USERNAME = "TU_USUARIO"
HF_TOKEN = "TU_HF_TOKEN"
BEST_CKPT = "runs/simcse_supervised_seed42/best_checkpoint"

!python -m src.export_and_publish \
    --checkpoint_dir {BEST_CKPT} \
    --export_dir ./st_best_model \
    --mode supervised \
    --push_to_hub \
    --repo_id f"{HF_USERNAME}/simcse-bert-base-snli" \
    --token {HF_TOKEN}